# Deepfake Detection: training on Google Colab

This notebook runs the full pipeline on a free Colab GPU:

1. Get the code
2. Download the **140k Real and Fake Faces** dataset
3. Isolate faces with MTCNN and build train / val / test splits
4. Train EfficientNet-B0 (two-stage transfer learning)
5. Evaluate on the held-out test set
6. Save the trained model and the report figures to Google Drive

**Before you start:** `Runtime -> Change runtime type -> T4 GPU`, then `Runtime -> Run all`.
Total time is roughly 45-75 minutes with the default sizes.

## 1. Check the GPU

In [ ]:
!nvidia-smi -L || echo "No GPU - switch the runtime to T4 GPU"

## 2. Get the code

If the GitHub repository is **public**, the first command just works.
If it is **private**, either make it public for tonight, or paste a GitHub personal access token in `TOKEN`.

In [ ]:
import os
TOKEN = ""  # only needed if the repo is private
BRANCH = "claude/bold-pasteur-mm917t"
url = f"https://{TOKEN}@github.com/rishabbachani/deepfake.git" if TOKEN else "https://github.com/rishabbachani/deepfake.git"
if not os.path.exists("/content/deepfake"):
    !git clone -q -b {BRANCH} {url} /content/deepfake
%cd /content/deepfake
!ls

In [ ]:
# Colab already has TensorFlow, OpenCV, scikit-learn, matplotlib and Flask.
!pip install -q "mtcnn>=1.0" kagglehub
import tensorflow as tf, keras
print("TensorFlow", tf.__version__, "| Keras", keras.__version__, "| GPU:", tf.config.list_physical_devices("GPU"))

## 3. Download the dataset

Uses `kagglehub`. If it asks for credentials: on kaggle.com go to *Settings -> API -> Create New Token*,
then run `import kagglehub; kagglehub.login()` and paste the username and key.

In [ ]:
import kagglehub
DATASET_PATH = kagglehub.dataset_download("xhlulu/140k-real-and-fake-faces")
print(DATASET_PATH)
!find {DATASET_PATH} -maxdepth 4 -type d | head -20

## 4. Build the face-crop dataset

Samples `TRAIN / VAL / TEST` images **per class** from the official splits, isolates the face with MTCNN
and writes 224x224 crops to `data/processed/`. Increase the sizes if you have time (each +5,000 per class adds a few minutes).

In [ ]:
TRAIN, VAL, TEST = 7000, 1500, 1500
!python prepare_dataset.py --source {DATASET_PATH} --train {TRAIN} --val {VAL} --test {TEST} --overwrite
!cat data/processed/summary.json

## 5. Train

Stage 1 trains only the new dense head on frozen ImageNet features; stage 2 fine-tunes the top 60 EfficientNet-B0 layers
with a 100x smaller learning rate. The checkpoint with the best validation ROC-AUC is kept.

In [ ]:
!python train.py --head-epochs 5 --finetune-epochs 10 --batch-size 32

## 6. Evaluate on the held-out test set

In [ ]:
!python evaluate.py

In [ ]:
from IPython.display import Image, display
for name in ["training_curves", "confusion_matrix", "roc_curve", "calibration"]:
    print(name); display(Image(f"reports/{name}.png", width=650))

## 7. Save everything

Copies `models/` and `reports/` to Google Drive (`MyDrive/deepfake_results`) and also downloads a zip.
Put the `models/` folder into the project folder on your laptop to run the web app.

In [ ]:
from google.colab import drive, files
drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/deepfake_results
!cp -r models reports data/processed/summary.json /content/drive/MyDrive/deepfake_results/
!zip -qr deepfake_results.zip models reports
files.download("deepfake_results.zip")

## 8. (Optional) Quick check of a single image

In [ ]:
!python predict.py data/processed/test/fake/$(ls data/processed/test/fake | head -1)
!python predict.py data/processed/test/real/$(ls data/processed/test/real | head -1)